# Result Visualisation Figures

Generates publication figures from:
- `final-runs/results_aggregates.csv`
- `final-runs/diagnostics_cross_release.csv`
- `final-runs/results_m2_top_services.csv`

All figures are saved to `figures/result_figures/` (no inline display).

In [40]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
import seaborn as sns
from pathlib import Path

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

plt.rcParams.update({
    'font.size': 9,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'legend.title_fontsize': 7,
    'legend.handlelength': 1.2,
    'legend.handleheight': 0.7,
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

OUT = Path('figures/result_figures')
OUT.mkdir(parents=True, exist_ok=True)
SAVED = []
WARNINGS = []

print('Output directory:', OUT.resolve())

Output directory: /Users/eoan/Sites/univaq/phd/energyAnalyser/controller/visualizations/figures/result_figures


## Load Data

In [53]:
df_agg = pd.read_csv('../final-runs/results_aggregates.csv')
df_cr  = pd.read_csv('../final-runs/diagnostics_cross_release.csv')
df_svc = pd.read_csv('../final-runs/results_m2_top_services.csv')
df_svc_m1 = pd.read_csv('../final-runs/results_m1_top_services.csv')

print('=== results_aggregates.csv ===')
print('Shape:', df_agg.shape)
print('Columns:', df_agg.columns.tolist())
print()
print('=== diagnostics_cross_release.csv ===')
print('Shape:', df_cr.shape)
print('Columns:', df_cr.columns.tolist())
print()
print('=== results_m2_top_services.csv ===')
print('Shape:', df_svc.shape)
print('Columns:', df_svc.columns.tolist())
print()
print('=== results_m1_top_services.csv ===')
print('Shape:', df_svc_m1.shape)
print('Columns:', df_svc_m1.columns.tolist())

=== results_aggregates.csv ===
Shape: (72, 27)
Columns: ['source_folder', 'app_name', 'machine', 'experiment_name', 'workload_level', 'run_count', 'energy_per_request_mean', 'energy_per_request_std', 'throughput_mean', 'throughput_std', 'p95_latency_mean', 'p95_latency_std', 'cpu_mean_mean', 'cpu_mean_std', 'energy_total_mean', 'energy_total_std', 'meter_power_mean', 'meter_power_std', 'meter_corrected_energy_wh_mean', 'meter_corrected_energy_wh_std', 'display_energy_total_mean', 'display_energy_total_std', 'display_energy_per_request_mean', 'display_energy_per_request_std', 'error_runs', 'runs_with_flags', 'total_runs']

=== diagnostics_cross_release.csv ===
Shape: (72, 14)
Columns: ['app_family', 'machine', 'workload_level', 'app_name', 'release_label', 'release_order', 'energy_total_mean', 'energy_total_pct_vs_oldest', 'top1_service', 'top1_mean_joules', 'top1_share_of_top5_pct', 'top2_service', 'top2_mean_joules', 'top2_share_of_top5_pct']

=== results_m2_top_services.csv ===
Shape

## Derive release metadata from app_name

In [54]:
FAMILY_MAP = {
    'onlineboutique':       ('onlineboutique', 'latest'),
    'onlineboutique-0.4.2': ('onlineboutique', '0.4.2'),
    'onlineboutique-0.9':   ('onlineboutique', '0.9'),
    'otel-demo':            ('otel-demo', 'latest'),
    'otel-demo-1.12.0':     ('otel-demo', '1.12.0'),
    'otel-demo-2.0.0':      ('otel-demo', '2.0.0'),
    'socialnetwork':        ('socialnetwork', 'latest'),
    'socialnetwork-0.1.0':  ('socialnetwork', '0.1.0'),
    'socialnetwork-0.2.0':  ('socialnetwork', '0.2.0'),
    'trainticket':          ('trainticket', 'latest'),
    'trainticket-0.0.4':    ('trainticket', '0.0.4'),
    'trainticket-0.2.0':    ('trainticket', '0.2.0'),
}

RELEASE_ORDER_MAP = {
    'onlineboutique': {'0.4.2': 0, '0.9': 1,    'latest': 2},
    'otel-demo':      {'1.12.0': 0, '2.0.0': 1, 'latest': 2},
    'socialnetwork':  {'0.1.0': 0,  '0.2.0': 1, 'latest': 2},
    'trainticket':    {'0.0.4': 0,  '0.2.0': 1, 'latest': 2},
}

def annotate_releases(df):
    df = df.copy()
    df['app_family']    = df['app_name'].map(lambda x: FAMILY_MAP.get(x, (x, 'unknown'))[0])
    df['release_label'] = df['app_name'].map(lambda x: FAMILY_MAP.get(x, (x, 'unknown'))[1])
    df['release_order'] = df.apply(
        lambda r: RELEASE_ORDER_MAP.get(r['app_family'], {}).get(r['release_label'], -1), axis=1)
    return df

df_agg = annotate_releases(df_agg)
df_svc = annotate_releases(df_svc)
df_svc_m1 = annotate_releases(df_svc_m1)

check = df_agg[['app_name','app_family','release_label','release_order']].drop_duplicates().sort_values(['app_family','release_order'])
print(check.to_string(index=False))

            app_name     app_family release_label  release_order
onlineboutique-0.4.2 onlineboutique         0.4.2              0
  onlineboutique-0.9 onlineboutique           0.9              1
      onlineboutique onlineboutique        latest              2
    otel-demo-1.12.0      otel-demo        1.12.0              0
     otel-demo-2.0.0      otel-demo         2.0.0              1
           otel-demo      otel-demo        latest              2
 socialnetwork-0.1.0  socialnetwork         0.1.0              0
 socialnetwork-0.2.0  socialnetwork         0.2.0              1
       socialnetwork  socialnetwork        latest              2
   trainticket-0.0.4    trainticket         0.0.4              0
   trainticket-0.2.0    trainticket         0.2.0              1
         trainticket    trainticket        latest              2


## Shared constants

In [43]:
APPS = ['onlineboutique', 'otel-demo', 'socialnetwork', 'trainticket']
WL_LABELS = ['low', 'medium', 'high']
WL_ORDER_MAP = {'low': 0, 'medium': 1, 'high': 2}

APP_DISPLAY = {
    'onlineboutique': 'Online Boutique',
    'otel-demo':      'OTel Demo',
    'socialnetwork':  'Social Network',
    'trainticket':    'Train Ticket',
}

# Release display labels per family
REL_DISPLAY = {
    'onlineboutique': {'0.4.2': 'v0.4.2', '0.9': 'v0.9',    'latest': 'v0.10.5 (latest)'},
    'otel-demo':      {'1.12.0': 'v1.12.0', '2.0.0': 'v2.0.0', 'latest': 'v2.2.0 (latest)'},
    'socialnetwork':  {'0.1.0': 'v0.1.0',  '0.2.0': 'v0.2.0', 'latest': 'latest'},
    'trainticket':    {'0.0.4': 'v0.0.4',  '0.2.0': 'v0.2.0', 'latest': 'latest'},
}

# Colours: oldest=lightest, latest=darkest
REL_COLORS = {0: '#aec7e8', 1: '#4a90d9', 2: '#1a4f8a'}

BAR_COLOR  = '#4a90d9'
LINE_COLOR = '#d95f02'

def rel_label(app_family, raw_label):
    return REL_DISPLAY.get(app_family, {}).get(raw_label, raw_label)

def vprefix(s):
    """Add v-prefix if the string starts with a digit."""
    return 'v' + s if s and s[0].isdigit() else s

def get_service_share(df_svc, app_family, machine, workload, target_service):
    """Return per-release share (%) of target_service within its run's top-5 attributed energy."""
    mask = ((df_svc['app_family'] == app_family) &
            (df_svc['machine'] == machine) &
            (df_svc['workload_level'] == workload))
    rows = []
    for _, grp in df_svc[mask].groupby('app_name'):
        total = grp['mean_joules'].sum()
        target = grp.loc[grp['service'] == target_service, 'mean_joules'].sum()
        rows.append({
            'release_label': grp['release_label'].iloc[0],
            'release_order': grp['release_order'].iloc[0],
            'share_pct': target / total * 100 if total > 0 else 0,
        })
    return pd.DataFrame(rows).sort_values('release_order')

print('Constants and helpers defined.')

Constants and helpers defined.


---
## Figure 1 — Energy per Request across Workload Levels

4-row × 2-column grid (rows = apps, columns = workstation / EC2).  
One line per release, y-axis log scale.

In [44]:
MACHINES = ['workstation', 'ec2']
MACHINE_DISPLAY = {'workstation': 'Workstation', 'ec2': 'EC2'}

fig, axes = plt.subplots(4, 2, figsize=(6.8, 9.5))
x_pos = [0, 1, 2]  # numeric positions for low / medium / high

for row, app in enumerate(APPS):
    for col, machine in enumerate(MACHINES):
        ax = axes[row, col]
        sub = df_agg[(df_agg['app_family'] == app) & (df_agg['machine'] == machine)].copy()
        sub['wl_order'] = sub['workload_level'].map(WL_ORDER_MAP)

        for rel_order in sorted(sub['release_order'].unique()):
            rs = sub[sub['release_order'] == rel_order].sort_values('wl_order')
            raw_lbl = rs['release_label'].iloc[0]
            label = rel_label(app, raw_lbl)
            color = REL_COLORS[rel_order]
            ax.plot(x_pos, rs['energy_per_request_mean'].values,
                    marker='o', markersize=4, linewidth=1.5,
                    color=color, label=label)

        ax.set_yscale('log')
        ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation(labelOnlyBase=False))
        ax.set_xticks(x_pos)
        ax.set_xticklabels(WL_LABELS, fontsize=8)
        ax.set_xlabel('Workload level', fontsize=8)
        ax.set_ylabel('Energy / request (J)', fontsize=8)
        ax.set_title(
            f'{APP_DISPLAY[app]}  ·  {MACHINE_DISPLAY[machine]}',
            fontsize=9, fontweight='bold')
        ax.legend(fontsize=6.5, loc='best')
        ax.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.6)

plt.tight_layout()
fig.savefig(OUT / 'figure_epr_workload.pdf')
fig.savefig(OUT / 'figure_epr_workload.png', dpi=150)
plt.close(fig)
SAVED += ['figure_epr_workload.pdf', 'figure_epr_workload.png']
print('Saved figure_epr_workload.pdf and .png')

Saved figure_epr_workload.pdf and .png


In [45]:
fig, axes = plt.subplots(4, 1, figsize=(3.2, 8.5))
x_pos = [0, 1, 2]

for row, app in enumerate(APPS):
    ax = axes[row]
    sub = df_agg[(df_agg['app_family'] == app) & (df_agg['machine'] == 'workstation')].copy()
    sub['wl_order'] = sub['workload_level'].map(WL_ORDER_MAP)

    for rel_order in sorted(sub['release_order'].unique()):
        rs = sub[sub['release_order'] == rel_order].sort_values('wl_order')
        label = rel_label(app, rs['release_label'].iloc[0])
        ax.plot(x_pos, rs['energy_per_request_mean'].values,
                marker='o', markersize=4, linewidth=1.5,
                color=REL_COLORS[rel_order], label=label)

    ax.set_yscale('log')
    ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation(labelOnlyBase=False))
    ax.set_xticks(x_pos)
    ax.set_xticklabels(WL_LABELS if row == 3 else [], fontsize=8)
    if row == 3:
        ax.set_xlabel('Workload level', fontsize=8)
    ax.set_ylabel('Energy / request (J)', fontsize=8)
    ax.set_title(APP_DISPLAY[app], fontsize=9, fontweight='bold')
    ax.legend(fontsize=6, loc='best')
    ax.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.6)

plt.tight_layout()
fig.savefig(OUT / 'figure_epr_workload_ws_only.pdf')
fig.savefig(OUT / 'figure_epr_workload_ws_only.png', dpi=150)
plt.close(fig)
SAVED += ['figure_epr_workload_ws_only.pdf', 'figure_epr_workload_ws_only.png']
print('Saved figure_epr_workload_ws_only.pdf and .png')

Saved figure_epr_workload_ws_only.pdf and .png


---
## Figure 2 — Cross-Release: Online Boutique

### Figure 2A — Workstation only, dual-axis grouped bars

In [46]:
ob_ws = df_cr[(df_cr['app_family'] == 'onlineboutique') & (df_cr['machine'] == 'workstation')].copy()
ob_ws['wl_order'] = ob_ws['workload_level'].map(WL_ORDER_MAP)

fig, axes = plt.subplots(1, 3, figsize=(6.8, 2.8))

for col, wl in enumerate(WL_LABELS):
    ax = axes[col]
    sub = ob_ws[ob_ws['workload_level'] == wl].sort_values('release_order')
    releases = [vprefix(r) for r in sub['release_label'].tolist()]
    x = np.arange(len(releases))

    ax.bar(x, sub['energy_total_mean'], color=BAR_COLOR, alpha=0.75, width=0.55,
           zorder=2, label='System energy (J)')
    ax.set_ylabel('System energy (J)' if col == 0 else '', fontsize=8)

    ax2 = ax.twinx()
    ax2.plot(x, sub['top1_share_of_top5_pct'], color=LINE_COLOR,
             marker='o', markersize=5, linewidth=1.5, zorder=3,
             label='cartservice share (%)')
    ax2.set_ylim(0, max(sub['top1_share_of_top5_pct'].max() * 1.35, 60))
    if col == 2:
        ax2.set_ylabel('cartservice share of top-5 (%)', fontsize=7)
    else:
        ax2.set_yticklabels([])

    ax.set_xticks(x)
    ax.set_xticklabels(releases, fontsize=7)
    ax.set_title(f'{wl.capitalize()} workload', fontsize=8)

    if col == 0:
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, fontsize=6, loc='upper right')

plt.tight_layout()
fig.savefig(OUT / 'figure_ob_crossrelease_ws_only.pdf')
plt.close(fig)
SAVED.append('figure_ob_crossrelease_ws_only.pdf')
print('Saved figure_ob_crossrelease_ws_only.pdf')

Saved figure_ob_crossrelease_ws_only.pdf


### Figure 2B — Workstation and EC2 side-by-side

In [47]:
fig, axes = plt.subplots(2, 3, figsize=(6.8, 5.2))

for row, machine in enumerate(['workstation', 'ec2']):
    ob_m = df_cr[
        (df_cr['app_family'] == 'onlineboutique') &
        (df_cr['machine'] == machine)
    ].copy()

    for col, wl in enumerate(WL_LABELS):
        ax = axes[row, col]
        sub = ob_m[ob_m['workload_level'] == wl].sort_values('release_order')
        releases = [vprefix(r) for r in sub['release_label'].tolist()]
        x = np.arange(len(releases))

        ax.bar(x, sub['energy_total_mean'], color=BAR_COLOR, alpha=0.75, width=0.55, zorder=2)

        ax2 = ax.twinx()
        ax2.plot(x, sub['top1_share_of_top5_pct'], color=LINE_COLOR,
                 marker='o', markersize=4, linewidth=1.5, zorder=3)
        ax2.set_ylim(0, sub['top1_share_of_top5_pct'].max() * 1.4)

        if machine == 'ec2':
            for xi, (_, rd) in zip(x, sub.iterrows()):
                ax2.annotate(
                    rd['top1_service'],
                    xy=(xi, rd['top1_share_of_top5_pct']),
                    xytext=(0, 7), textcoords='offset points',
                    ha='center', fontsize=4.5, rotation=25, color=LINE_COLOR)

        ax.set_xticks(x)
        ax.set_xticklabels(releases, fontsize=7)

        m_label = 'Workstation' if machine == 'workstation' else 'EC2'
        ax.set_title(f'{m_label} — {wl.capitalize()}', fontsize=8)

        if col == 0:
            ax.set_ylabel('System energy (J)', fontsize=8)
        if col == 2:
            ax2.set_ylabel('Top-1 service share of top-5 (%)', fontsize=6.5)
        else:
            ax2.set_yticklabels([])

bar_patch = Patch(color=BAR_COLOR, alpha=0.75, label='System energy (J)')
line_patch = plt.Line2D([0], [0], color=LINE_COLOR, marker='o', markersize=4,
                         linewidth=1.5, label='Top-1 service share (%)')
fig.legend(handles=[bar_patch, line_patch], fontsize=7,
           loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(OUT / 'figure_ob_crossrelease_both_envs.pdf')
plt.close(fig)
SAVED.append('figure_ob_crossrelease_both_envs.pdf')
print('Saved figure_ob_crossrelease_both_envs.pdf')

Saved figure_ob_crossrelease_both_envs.pdf


### Figure 2C — Stacked area chart, workstation only, medium workload

In [48]:
ob_ws_med = df_svc[
    (df_svc['app_family'] == 'onlineboutique') &
    (df_svc['machine'] == 'workstation') &
    (df_svc['workload_level'] == 'medium')
].copy().sort_values('release_order')

pivot = ob_ws_med.pivot_table(
    index='release_order', columns='service',
    values='mean_joules', fill_value=0)

rl_map = ob_ws_med.drop_duplicates('release_order').set_index('release_order')['release_label']
x_labels = [vprefix(rl_map[i]) for i in pivot.index]
x = np.arange(len(pivot))

services = pivot.columns.tolist()
pal = sns.color_palette('Set2', len(services))
svc_colors_2c = dict(zip(services, pal))

fig, ax = plt.subplots(figsize=(4.0, 3.0))

vals   = [pivot[svc].values for svc in services]
colors = [svc_colors_2c[svc] for svc in services]
ax.stackplot(x, vals, labels=services, colors=colors, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=8)
ax.set_xlabel('Release', fontsize=8)
ax.set_ylabel('Attributed energy (J)', fontsize=8)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], fontsize=6,
          bbox_to_anchor=(1.01, 1), loc='upper left',
          title='Service', title_fontsize=6)
ax.grid(True, axis='y', linestyle='--', linewidth=0.4, alpha=0.6)

plt.tight_layout()
fig.savefig(OUT / 'figure_ob_crossrelease_stacked.pdf')
plt.close(fig)
SAVED.append('figure_ob_crossrelease_stacked.pdf')
print('Saved figure_ob_crossrelease_stacked.pdf')

Saved figure_ob_crossrelease_stacked.pdf


---
## Figure 3 — Cross-Release: OTel Demo

**Note on top-1 service:** For OTel Demo, `load-generator` is always the top-1 attributed service on both environments (it is a component within the demo deployment). The task specifies `product-catalog` (workstation) and `frontend` (EC2) as the services of interest for the right axis. These shares are computed from `results_m2_top_services.csv`.

### Figure 3A — Workstation only, dual-axis

In [49]:
otel_ws = df_cr[
    (df_cr['app_family'] == 'otel-demo') &
    (df_cr['machine'] == 'workstation')
].copy()

fig, axes = plt.subplots(1, 3, figsize=(6.8, 2.8))

for col, wl in enumerate(WL_LABELS):
    ax = axes[col]
    sub = otel_ws[otel_ws['workload_level'] == wl].sort_values('release_order')
    releases = [vprefix(r) for r in sub['release_label'].tolist()]
    x = np.arange(len(releases))

    ax.bar(x, sub['energy_total_mean'], color=BAR_COLOR, alpha=0.75, width=0.55,
           zorder=2, label='System energy (J)')
    ax.set_ylabel('System energy (J)' if col == 0 else '', fontsize=8)

    pc_share = get_service_share(df_svc, 'otel-demo', 'workstation', wl, 'product-catalog')

    ax2 = ax.twinx()
    ax2.plot(
        np.arange(len(pc_share)),
        pc_share['share_pct'].values,
        color=LINE_COLOR, marker='s', markersize=5, linewidth=1.5,
        zorder=3, label='product-catalog share (%)')
    ax2.set_ylim(0, max(pc_share['share_pct'].max() * 1.4, 30))
    if col == 2:
        ax2.set_ylabel('product-catalog share of top-5 (%)', fontsize=7)
    else:
        ax2.set_yticklabels([])

    ax.set_xticks(x)
    ax.set_xticklabels(releases, fontsize=7)
    ax.set_title(f'{wl.capitalize()} workload', fontsize=8)

    if col == 0:
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, fontsize=6, loc='upper right')

plt.tight_layout()
fig.savefig(OUT / 'figure_otel_crossrelease_ws_only.pdf')
plt.close(fig)
SAVED.append('figure_otel_crossrelease_ws_only.pdf')
print('Saved figure_otel_crossrelease_ws_only.pdf')

Saved figure_otel_crossrelease_ws_only.pdf


### Figure 3B — Workstation and EC2 side-by-side

In [50]:
TARGET_SVC = {'workstation': 'product-catalog', 'ec2': 'frontend'}

fig, axes = plt.subplots(2, 3, figsize=(6.8, 5.2))

for row, machine in enumerate(['workstation', 'ec2']):
    otel_m = df_cr[
        (df_cr['app_family'] == 'otel-demo') &
        (df_cr['machine'] == machine)
    ].copy()

    for col, wl in enumerate(WL_LABELS):
        ax = axes[row, col]
        sub = otel_m[otel_m['workload_level'] == wl].sort_values('release_order')
        releases = [vprefix(r) for r in sub['release_label'].tolist()]
        x = np.arange(len(releases))

        ax.bar(x, sub['energy_total_mean'], color=BAR_COLOR, alpha=0.75, width=0.55, zorder=2)

        target_svc = TARGET_SVC[machine]
        svc_share = get_service_share(df_svc, 'otel-demo', machine, wl, target_svc)

        ax2 = ax.twinx()
        ax2.plot(
            np.arange(len(svc_share)),
            svc_share['share_pct'].values,
            color=LINE_COLOR, marker='s', markersize=4, linewidth=1.5, zorder=3)
        ax2.set_ylim(0, max(svc_share['share_pct'].max() * 1.4, 30))

        ax.set_xticks(x)
        ax.set_xticklabels(releases, fontsize=7)

        m_label = 'Workstation' if machine == 'workstation' else 'EC2'
        ax.set_title(f'{m_label} — {wl.capitalize()}', fontsize=8)

        if col == 0:
            ax.set_ylabel('System energy (J)', fontsize=8)
        if col == 2:
            ax2.set_ylabel(f'{target_svc} share of top-5 (%)', fontsize=6.5)
        else:
            ax2.set_yticklabels([])

bar_patch  = Patch(color=BAR_COLOR, alpha=0.75, label='System energy (J)')
line_ws    = plt.Line2D([0], [0], color=LINE_COLOR, marker='s', markersize=4,
                         linewidth=1.5, label='product-catalog share (WS, %)')
line_ec2   = plt.Line2D([0], [0], color=LINE_COLOR, marker='s', markersize=4,
                         linewidth=1.5, linestyle='--', label='frontend share (EC2, %)')
fig.legend(handles=[bar_patch, line_ws, line_ec2], fontsize=6.5,
           loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(OUT / 'figure_otel_crossrelease_both_envs.pdf')
plt.close(fig)
SAVED.append('figure_otel_crossrelease_both_envs.pdf')
print('Saved figure_otel_crossrelease_both_envs.pdf')

Saved figure_otel_crossrelease_both_envs.pdf


---
## Figure 4 — Service Concentration

Medium workload, workstation only. Horizontal stacked bar chart per app family.  
Each bar = one release; segments = top-5 services by mean attributed energy.

> **Note on 'other' segment:** `results_m2_top_services.csv` contains exactly the top-5 services per run. The system-level `energy_total_mean` in `results_aggregates.csv` uses a different measurement methodology (net wall-power delta) and is not directly comparable in magnitude to the per-service CPU attribution joules, so a reliable 'other' segment cannot be computed from the available data. The figure therefore shows the top-5 services only and their relative composition is still informative.

In [55]:
def make_service_concentration_outputs(df_services, stem):
    svc_ws_med = df_services[
        (df_services['machine'] == 'workstation') &
        (df_services['workload_level'] == 'medium')
    ].copy()

    # Exclude load-generator from OTel Demo (infrastructure component, not an app service)
    svc_ws_med = svc_ws_med[
        ~((svc_ws_med['app_family'] == 'otel-demo') & (svc_ws_med['service'] == 'load-generator'))
    ]

    # Build global service -> colour mapping (consistent across all subplots for this model)
    all_svcs_ranked = (
        svc_ws_med.groupby('service')['mean_joules']
        .sum()
        .sort_values(ascending=False)
        .index.tolist()
    )
    global_pal = sns.color_palette('tab20', min(len(all_svcs_ranked), 20))
    global_svc_color = {svc: global_pal[i % 20] for i, svc in enumerate(all_svcs_ranked)}

    # Keep the cartservice highlight for comparability with the M2 figure
    global_svc_color['cartservice'] = '#d62728'
    other_color = '#cccccc'

    def draw_panel(ax, app, show_title=True):
        app_data = svc_ws_med[svc_ws_med['app_family'] == app].copy()
        app_data = app_data.sort_values('release_order')

        rl_map = (app_data.drop_duplicates('release_order')
                  .set_index('release_order')['release_label'])
        release_orders = sorted(app_data['release_order'].unique())
        y_labels = [vprefix(rl_map[i]) for i in release_orders]
        y_pos = np.arange(len(release_orders))

        top3 = (app_data.groupby('service')['mean_joules']
                .sum()
                .sort_values(ascending=False)
                .head(3)
                .index.tolist())

        left = np.zeros(len(release_orders))

        for svc in top3:
            vals = np.array([
                app_data.loc[(app_data['release_order'] == ro) & (app_data['service'] == svc),
                             'mean_joules'].sum()
                for ro in release_orders
            ])
            color = global_svc_color.get(svc, '#888888')
            ax.barh(y_pos, vals, left=left, color=color, label=svc, height=0.55)
            left += vals

        other_vals = np.array([
            app_data.loc[(app_data['release_order'] == ro) & (~app_data['service'].isin(top3)),
                         'mean_joules'].sum()
            for ro in release_orders
        ])
        ax.barh(y_pos, other_vals, left=left, color=other_color, label='Other', height=0.55)

        ax.set_yticks(y_pos)
        ax.set_yticklabels(y_labels, fontsize=8)
        ax.set_xlabel('Attributed energy (J)', fontsize=8)
        if show_title:
            ax.set_title(APP_DISPLAY[app], fontsize=9, fontweight='bold')
        ax.grid(True, axis='x', linestyle='--', linewidth=0.4, alpha=0.6)

        ax.legend(fontsize=6.5, loc='center left',
                  bbox_to_anchor=(1.01, 0.5),
                  title='Service', title_fontsize=6.5,
                  frameon=True, edgecolor='#cccccc')

    # Combined 2x2 figure
    fig, axes = plt.subplots(2, 2, figsize=(9.0, 5.5))
    for idx, app in enumerate(APPS):
        draw_panel(axes[idx // 2, idx % 2], app)

    plt.tight_layout()
    combined_pdf = f'{stem}.pdf'
    combined_png = f'{stem}.png'
    fig.savefig(OUT / combined_pdf)
    fig.savefig(OUT / combined_png, dpi=150)
    plt.close(fig)

    # Individual per-app PDFs (titleless)
    per_app_files = []
    for app in APPS:
        fig_single, ax_single = plt.subplots(1, 1, figsize=(4.6, 2.3))
        draw_panel(ax_single, app, show_title=False)
        plt.tight_layout()
        out_name = f'{stem}_{app}.pdf'
        fig_single.savefig(OUT / out_name)
        plt.close(fig_single)
        per_app_files.append(out_name)

    return [combined_pdf, combined_png] + per_app_files

# NOTE on interpretation: M2 is the primary valid service attribution model; M1 is diagnostic.
saved_m2 = make_service_concentration_outputs(df_svc, 'figure_service_concentration_v2')
saved_m1 = make_service_concentration_outputs(df_svc_m1, 'figure_service_concentration_v2_m1')

SAVED += saved_m2 + saved_m1
print('Saved M2 and M1 service concentration figure sets')

Saved M2 and M1 service concentration figure sets


---
## Summary

In [52]:
print('=' * 60)
print('FILES SAVED')
print('=' * 60)
for f in SAVED:
    path = OUT / f
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  [{status}] {f}  ({size_kb:.1f} KB)')

if WARNINGS:
    print()
    print('=' * 60)
    print('WARNINGS')
    print('=' * 60)
    for w in WARNINGS:
        print(f'  ! {w}')
else:
    print('\nNo warnings.')

FILES SAVED
  [OK] figure_epr_workload.pdf  (39.1 KB)
  [OK] figure_epr_workload.png  (218.3 KB)
  [OK] figure_epr_workload_ws_only.pdf  (30.5 KB)
  [OK] figure_epr_workload_ws_only.png  (102.0 KB)
  [OK] figure_ob_crossrelease_ws_only.pdf  (18.8 KB)
  [OK] figure_ob_crossrelease_both_envs.pdf  (23.3 KB)
  [OK] figure_ob_crossrelease_stacked.pdf  (15.1 KB)
  [OK] figure_otel_crossrelease_ws_only.pdf  (18.5 KB)
  [OK] figure_otel_crossrelease_both_envs.pdf  (22.0 KB)
  [OK] figure_service_concentration_v2.pdf  (27.6 KB)
  [OK] figure_service_concentration_v2.png  (61.3 KB)
  [OK] figure_service_concentration_v2_onlineboutique.pdf  (14.2 KB)
  [OK] figure_service_concentration_v2_otel-demo.pdf  (15.1 KB)
  [OK] figure_service_concentration_v2_socialnetwork.pdf  (15.1 KB)
  [OK] figure_service_concentration_v2_trainticket.pdf  (14.6 KB)

No warnings.
